In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px



In [23]:
# Plot settings
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 120

In [24]:
# Load data
markets = pd.read_csv(
    '/home/diegom/Proyectos-DS/Polymarket_Prediction_Markets/polymarket_markets.csv',
    low_memory=False
)

events = pd.read_csv(
    '/home/diegom/Proyectos-DS/Polymarket_Prediction_Markets/polymarket_events.csv',
    low_memory=False
)

In [25]:
# Volume percentiles to decide liquidity threshold
markets['volume'].quantile([0.25,0.50, 0.75, 0.90, 0.95, 0.99]).round(2)

0.25        236.55
0.50       3832.24
0.75      24610.42
0.90      86806.27
0.95     185502.79
0.99    1392451.40
Name: volume, dtype: float64

In [26]:
# Define liquidity target — top 25% by volume
LIQUIDITY_THRESHOLD = markets['volume'].quantile(0.75)

markets['high_liquidity'] = (markets['volume'] >= LIQUIDITY_THRESHOLD).astype(int)

In [27]:
# Check class balance
markets['high_liquidity'].value_counts(normalize=True).round(3)

high_liquidity
0    0.766
1    0.234
Name: proportion, dtype: float64

In [28]:
# Parse timestamps
markets['createdAt'] = pd.to_datetime(markets['createdAt'], format='ISO8601')
markets['endDate'] = pd.to_datetime(markets['endDate'], format='ISO8601')
markets['startDate'] = pd.to_datetime(markets['startDate'], format='ISO8601')

In [29]:
# --- TEMPORAL FEATURES ---
# Market duration in days (known at creation)
markets['duration_days'] = (markets['endDate'] - markets['startDate']).dt.days

# Hour and day of week of market creation (captures platform activity patterns)
markets['created_hour'] = markets['createdAt'].dt.hour
markets['created_dow'] = markets['createdAt'].dt.dayofweek
markets['created_month'] = markets['createdAt'].dt.month

markets[['duration_days', 'created_hour', 'created_dow', 'created_month']].describe().round(2)

,duration_days,created_hour,created_dow,created_month
count,97219.00,100795.00,100795.00,100795.00
mean,40.78,14.28,2.51,10.04
std,113.88,6.46,1.99,2.10
min,-1420.00,0.00,0.00,1.00
25%,0.00,10.00,1.00,10.00
50%,4.00,16.00,2.00,11.00
75%,13.00,19.00,4.00,11.00
max,1214.00,23.00,6.00,12.00


In [30]:
# --- STRUCTURAL FEATURES ---
markets['spread_initial'] = markets['spread']

markets['has_orderbook'] = markets['enableOrderBook'].fillna('false').astype(str).map(
    lambda x: 1 if x.lower() == 'true' else 0
)

markets['is_featured'] = markets['featured'].fillna('false').astype(str).map(
    lambda x: 1 if x.lower() == 'true' else 0
)

markets['is_accepting_orders'] = markets['acceptingOrders'].fillna('false').astype(str).map(
    lambda x: 1 if x.lower() == 'true' else 0
)

markets['is_negrisk'] = markets['negRisk'].fillna('false').astype(str).map(
    lambda x: 1 if x.lower() == 'true' else 0
)

markets[['spread_initial', 'has_orderbook', 'is_featured', 'is_accepting_orders', 'is_negrisk']].describe().round(3)

/tmp/ipykernel_54128/2573011157.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  markets['is_accepting_orders'] = markets['acceptingOrders'].fillna('false').astype(str).map(
/tmp/ipykernel_54128/2573011157.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  markets['is_negrisk'] = markets['negRisk'].fillna('false').astype(str).map(


,spread_initial,has_orderbook,is_featured,is_accepting_orders,is_negrisk
count,100795.000,100795.000,100795.000,100795.000,100795.000
mean,0.234,0.953,0.000,0.262,0.288
std,0.394,0.211,0.021,0.440,0.453
min,0.001,0.000,0.000,0.000,0.000
25%,0.001,1.000,0.000,0.000,0.000
50%,0.010,1.000,0.000,0.000,0.000
75%,0.180,1.000,0.000,1.000,1.000
max,1.000,1.000,1.000,1.000,1.000


In [31]:
# --- CATEGORY FEATURES ---
# Encode top categories, group rest as 'other'
top_categories = markets['category'].value_counts().head(10).index.tolist()

markets['category_clean'] = markets['category'].apply(
    lambda x: x if x in top_categories else 'other'
)

# Category encoding as dummies
category_dummies = pd.get_dummies(markets['category_clean'], prefix='cat').astype(int)

markets = pd.concat([markets, category_dummies], axis=1)

markets['category_clean'].value_counts()

/tmp/ipykernel_54128/845696936.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  markets['category_clean'] = markets['category'].apply(


category_clean
other                 96660
Sports                 2540
Crypto                  374
US-current-affairs      339
Pop-Culture             218
Coronavirus             153
Business                137
NBA Playoffs            126
NFTs                    110
Chess                    77
Art                      61
Name: count, dtype: int64

In [32]:
# --- EVENT-LEVEL FEATURES ---
# Number of markets per event (proxy for event complexity/importance)
event_market_count = markets.groupby('event_id').size().rename('event_market_count')
markets = markets.join(event_market_count, on='event_id')

# Total event volume (how important is the parent event)
event_volume = markets.groupby('event_id')['volume'].sum().rename('event_total_volume')
markets = markets.join(event_volume, on='event_id')

markets[['event_market_count', 'event_total_volume']].describe().round(2)

,event_market_count,event_total_volume
count,100795.00,1.007950e+05
mean,13.25,2.032974e+06
std,20.82,1.922057e+07
min,1.00,0.000000e+00
25%,1.00,3.312350e+03
50%,5.00,2.889085e+04
75%,15.00,2.006078e+05
max,139.00,6.086758e+08


In [33]:
# --- USE EVENT CATEGORY INSTEAD ---
# category in markets is mostly null, use parent event category
event_category = events.set_index('id')['category'].rename('event_category')
markets = markets.join(event_category, on='event_id')

top_event_cats = markets['event_category'].value_counts().head(10).index.tolist()
markets['event_category_clean'] = markets['event_category'].apply(
    lambda x: x if x in top_event_cats else 'other'
)

event_cat_dummies = pd.get_dummies(markets['event_category_clean'], prefix='ecat').astype(int)
markets = pd.concat([markets, event_cat_dummies], axis=1)

markets['event_category_clean'].value_counts()

/tmp/ipykernel_54128/721734476.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  markets['event_category_clean'] = markets['event_category'].apply(


event_category_clean
other                 96778
Sports                 2508
Crypto                  350
US-current-affairs      309
Pop-Culture             205
Coronavirus             149
NBA Playoffs            124
Business                121
NFTs                    114
Chess                    76
Art                      61
Name: count, dtype: int64

In [34]:
# Check how many events have null category
events['category'].isna().sum(), events['category'].nunique()

(np.int64(41014), 19)

In [35]:
# Maybe tags column has better coverage
events[['category', 'tags']].head(10)

,category,tags
0,NaN,"[{""id"": ""2"", ""label"": ""Politics"", ""slug"": ""pol..."
1,NaN,"[{""id"": ""101438"", ""label"": ""Romania"", ""slug"": ..."
2,NaN,"[{""id"": ""1"", ""label"": ""Sports"", ""slug"": ""sport..."
3,NaN,"[{""id"": ""21"", ""label"": ""Crypto"", ""slug"": ""cryp..."
4,NaN,"[{""id"": ""1"", ""label"": ""Sports"", ""slug"": ""sport..."
5,NaN,"[{""id"": ""1597"", ""label"": ""Global Elections"", ""..."
6,NaN,"[{""id"": ""596"", ""label"": ""Culture"", ""slug"": ""po..."
7,NaN,"[{""id"": ""2"", ""label"": ""Politics"", ""slug"": ""pol..."
8,NaN,"[{""id"": ""235"", ""label"": ""Bitcoin"", ""slug"": ""bi..."
9,NaN,"[{""id"": ""596"", ""label"": ""Culture"", ""slug"": ""po..."


In [36]:
import json

# Extract first tag label from JSON string
def extract_first_tag(tags_str):
    try:
        tags = json.loads(tags_str)
        if tags and len(tags) > 0:
            return tags[0]['label']
    except:
        pass
    return 'other'

events['tag_category'] = events['tags'].fillna('[]').apply(extract_first_tag)
events['tag_category'].value_counts().head(15)

tag_category
Up or Down       17851
Sports            7660
Crypto            5388
All               4199
Esports           1320
Tennis             688
Politics           440
Crypto Prices      349
Games              327
Hide From New      303
Basketball         266
NCAA               207
Ethereum           196
Earnings           175
Solana             166
Name: count, dtype: int64

In [37]:
# Keep Up or Down as a valid category, just remove internal platform tags
internal_tags = ['All', 'Hide From New']

events['tag_category_clean'] = events['tag_category'].apply(
    lambda x: 'other' if x in internal_tags else x
)

top_tags = events['tag_category_clean'].value_counts().head(10).index.tolist()
events['tag_category_clean'] = events['tag_category_clean'].apply(
    lambda x: x if x in top_tags else 'other'
)

events['tag_category_clean'].value_counts()

tag_category_clean
Up or Down       17851
other             9551
Sports            7660
Crypto            5388
Esports           1320
Tennis             688
Politics           440
Crypto Prices      349
Games              327
Basketball         266
Name: count, dtype: int64

In [38]:
# Join tag category to markets via event_id
event_tag_cat = events.set_index('id')['tag_category_clean'].rename('tag_category')
markets = markets.join(event_tag_cat, on='event_id')

# Encode as dummies
tag_dummies = pd.get_dummies(markets['tag_category'], prefix='tag').astype(int)
markets = pd.concat([markets, tag_dummies], axis=1)

markets['tag_category'].value_counts()

tag_category
other            38570
Sports           28777
Up or Down       17851
Crypto            6137
Politics          3360
Esports           2666
Games             1377
Tennis             867
Basketball         718
Crypto Prices      472
Name: count, dtype: int64

In [39]:
# --- BUILD CLEAN FEATURE MATRIX ---
feature_cols = [
    # Target
    'high_liquidity',
    
    # Temporal
    'duration_days', 'created_hour', 'created_dow', 'created_month',
    
    # Structural
    'spread_initial', 'has_orderbook', 'is_featured',
    'is_accepting_orders', 'is_negrisk',
    
    # Event-level
    'event_market_count', 'event_total_volume',
    
    # Category dummies
] + [c for c in markets.columns if c.startswith('tag_')]

df = markets[feature_cols].copy()

# Drop rows with null duration (bad timestamps)
df = df[df['duration_days'] >= 0]

# Fill remaining nulls
df = df.fillna(0)

df.shape

(94204, 23)

In [40]:
# Final feature matrix overview
df.describe().round(3)

,high_liquidity,duration_days,created_hour,created_dow,created_month,spread_initial,has_orderbook,is_featured,is_accepting_orders,is_negrisk,...,tag_Basketball,tag_Crypto,tag_Crypto Prices,tag_Esports,tag_Games,tag_Politics,tag_Sports,tag_Tennis,tag_Up or Down,tag_other
count,94204.000,94204.000,94204.000,94204.000,94204.000,94204.000,94204.000,94204.000,94204.000,94204.000,...,94204.000,94204.000,94204.000,94204.000,94204.000,94204.000,94204.000,94204.000,94204.000,94204.000
mean,0.241,42.235,14.214,2.502,10.202,0.195,0.983,0.000,0.258,0.287,...,0.008,0.064,0.005,0.028,0.014,0.032,0.302,0.009,0.189,0.348
std,0.428,115.093,6.474,2.002,1.836,0.363,0.129,0.022,0.438,0.452,...,0.087,0.245,0.071,0.165,0.119,0.176,0.459,0.095,0.392,0.476
min,0.000,0.000,0.000,0.000,1.000,0.001,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,0.000,1.000,10.000,1.000,10.000,0.001,1.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
50%,0.000,4.000,16.000,2.000,11.000,0.010,1.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
75%,0.000,13.000,19.000,4.000,11.000,0.080,1.000,0.000,1.000,1.000,...,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,1.000
max,1.000,1214.000,23.000,6.000,12.000,1.000,1.000,1.000,1.000,1.000,...,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000


In [41]:
# Class balance in clean dataset
df['high_liquidity'].value_counts(normalize=True).round(3)

high_liquidity
0    0.759
1    0.241
Name: proportion, dtype: float64

In [42]:
# Save processed feature matrix
df.to_parquet(
    '/home/diegom/Proyectos-DS/Polymarket_Prediction_Markets/data/features.parquet',
    index=False
)